<a href="https://colab.research.google.com/github/jasirard/DSC563_M03/blob/main/sirard_jeffrey_M03_Preparation_and_Self_Checks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modules 3 and 4: Exploration, Chart Repair, and Accessible Design
Use **this one notebook for both weeks**. Save your own copy now (**File → Save a copy in Drive**) and keep working in that copy.

| Week | What you do | What you submit |
|---|---|---|
| Module 3 | Explore five charts, practice checking numbers, and plan two revisions (Phase A). | Nothing. Save your work. |
| Module 4 | Make two before/after revisions, including one for accessibility, and check two numbers (Phase B). | One 3–4-minute VoiceThread (180 points), due Sunday of Module 4. |

### How this notebook works
- **Part 1 (everyone):** you practice on workshop data built from your assigned integer. Enter your integer in Step 1 and run the cells in order.
- **Part 2 (DSC 563 required · BUS/DSC 363 optional for bonus points):** after Part 1, you load your own dataset from Google Drive and apply the same techniques to it.
- **Run only** means run the cell without changes. **Edit here** means change the marked settings, then run.
- **Tableau users:** run Steps 1–4 to download your CSV, then do the same tasks in Tableau.

### Glossary
- **Student integer:** your assigned number (1–200). It builds your dataset, and the same number always gives the same data. 0 is for practice.
- **Grain:** what one row represents. Here, one row is one workshop session.
- **Utilization:** attendance ÷ available seats × 100.
- **Paired denominator:** when you calculate a rate, count seats only for sessions where attendance is known. Unknown attendance is not zero.
- **Coverage:** known records out of all records, such as 360/380 sessions.
- **Percentage point (pp):** the difference between two percentages. 60% to 65% is +5 pp.
- **Seat-weighted vs. equal-session average:** a seat-weighted rate lets large sessions count more; an equal-session average counts every session the same.
- **Reference key:** a label such as `group:North` or `rate:East` that names a number calculated from the data, so you can check a number shown in your chart.

### Workshop data dictionary
The data are **simulated** workshop sessions. Month numbers are periods 1–24, not calendar dates.

| Field | Meaning |
|---|---|
| `session_id` | Session identifier |
| `month` | Period number, 1–24 |
| `neighborhood` | One of six service areas (for online sessions, the area served) |
| `topic` | One of four workshop topics |
| `delivery_format` | In person or online |
| `schedule` | Weekday daytime, weekday evening, or weekend |
| `duration_hours` | Session length in hours |
| `capacity_seats` | Available seats |
| `registrations` | Confirmed registrations (never more than capacity) |
| `attendance_visits` | Recorded attendance; blank means unknown, not zero |
| `waitlist_count` | Requests beyond the available seats |
| `operating_cost_usd` | Session cost in US dollars |
| `survey_response_count` | Attendees who answered the survey |
| `mean_satisfaction` | Average rating from survey responders, 1–5 |

Attendance counts seat visits, not people: one person who attends three sessions counts three times.

# Part 1: Course data (all students)
## Phase A: Module 3 exploration and planning
Complete Steps 1–7 in Module 3. Use the Module 3 readings as you explore.

## Step 1: Settings (Edit here)
1. Replace `0` with your assigned integer. Use the same number every week.
2. Describe your audience and the question your charts should answer.
3. Pick two neighborhoods to compare over time in `COMPARE_CATEGORIES`.

In [1]:
# STEP 1: SETTINGS. Edit the four values below, then run.
STUDENT_INTEGER = 23  # Your assigned integer, 1-200 (0 = practice).
AUDIENCE = 'Workshop coordinator'
QUESTION = 'How do attendance, utilization, and recorded demand affect planning?'
COMPARE_CATEGORIES = ['East', 'North']  # Two of: North, East, South, West, Central, Riverside.

# Tools used in this notebook.
from pathlib import Path
import json, math, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Check the integer, make a folder for your files, and set a clean chart style.
if isinstance(STUDENT_INTEGER, bool) or not isinstance(STUDENT_INTEGER, int) or not 0 <= STUDENT_INTEGER <= 200:
    raise ValueError('Enter a whole number from 1 to 200, or 0 for practice.')
OUT = Path(f'modules_3_4_integer_{STUDENT_INTEGER}')
OUT.mkdir(exist_ok=True)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11, 'axes.spines.top': False,
                     'axes.spines.right': False, 'savefig.bbox': 'tight'})
SOURCE = 'Simulated workshop sessions (course data)'
VALUE_LABEL = 'Attendance (seat visits)'
RATE_LABEL = 'Seat utilization (%)'
print(f'Integer {STUDENT_INTEGER}. Your files are saved in the folder {OUT}/')
if STUDENT_INTEGER == 0:
    print('PRACTICE DATA: enter your assigned integer before you make charts you will submit.')

Integer 23. Your files are saved in the folder modules_3_4_integer_23/


## Step 2: Build your data (Run only)
This cell creates the workshop records for your integer. Don't change it.

In [2]:
# STEP 2: BUILD YOUR WORKSHOP DATA. Run only; do not edit this cell.
# Your integer always produces the same records, so your work can continue week to week.
def make_workshop_data(integer):
    # A fresh PCG64 generator makes repeated runs with this integer repeat the records.
    rng = np.random.Generator(np.random.PCG64(integer))
    neighborhoods = ['North', 'East', 'South', 'West', 'Central', 'Riverside']
    topics = ['Digital skills', 'Career skills', 'Creative practice', 'Personal finance']
    formats = ['In person', 'Online']
    schedules = ['Weekday daytime', 'Weekday evening', 'Weekend']
    # Neighborhood and topic differences create comparisons without implying real findings.
    area_effect = dict(zip(neighborhoods, rng.normal(0, .08, len(neighborhoods))))
    topic_effect = dict(zip(topics, rng.normal(0, .09, len(topics))))
    rows = []
    for month in range(1, 25):
        cycle = .08 * np.sin(2 * np.pi * (month - 1) / 12)
        for area in neighborhoods:
            for topic_index, topic in enumerate(topics):
                for delivery in formats:
                    # Between one and three sessions per combination varies group sizes.
                    for repeat in range(int(rng.integers(1, 4))):
                        schedule = str(rng.choice(schedules, p=[.28, .47, .25]))
                        duration = float(rng.choice([1., 1.5, 2., 2.5, 3.]))
                        capacity = int(rng.integers(18, 81) if delivery == 'In person' else rng.integers(45, 181))
                        demand = (.74 + area_effect[area] + topic_effect[topic] + cycle
                                  + .004 * (month - 12) + (.09 if schedule == 'Weekday evening' else -.02)
                                  + (.08 if delivery == 'Online' else 0) + rng.normal(0, .18))
                        requests = max(0, int(round(capacity * demand)))
                        registrations = min(capacity, requests)
                        waitlist = max(0, requests - capacity)
                        # Registrations are confirmed places; this exercise assumes no walk-ins.
                        show_probability = float(np.clip(.85 + (.05 if delivery == 'In person' else -.07)
                                                        - .025 * (duration - 1) + rng.normal(0, .05), .45, .99))
                        attendance = int(rng.binomial(registrations, show_probability))
                        cost = round(max(20., 90 + 65 * duration + (.8 if delivery == 'In person' else .25) * capacity
                                         + 16 * topic_index + rng.normal(0, 23)), 2)
                        response_probability = float(np.clip(.40 + (.12 if delivery == 'In person' else -.12)
                                                             + rng.normal(0, .06), .05, .85))
                        responses = int(rng.binomial(attendance, response_probability))
                        # Ratings summarize responders only; they do not measure all attendees.
                        satisfaction = (round(float(np.clip(3.8 + .15 * topic_index + .07 * (duration - 2)
                                                           + rng.normal(0, .35), 1, 5)), 2)
                                        if responses else np.nan)
                        rows.append({'session_id': f'WS-{len(rows)+1:05d}', 'month': month,
                                     'neighborhood': area, 'topic': topic, 'delivery_format': delivery,
                                     'schedule': schedule, 'duration_hours': duration,
                                     'capacity_seats': capacity, 'registrations': registrations,
                                     'attendance_visits': float(attendance), 'waitlist_count': waitlist,
                                     'operating_cost_usd': cost, 'survey_response_count': float(responses),
                                     'mean_satisfaction': satisfaction})
    frame = pd.DataFrame(rows)
    # Missing reports are more common in East; missing attendance is never replaced by zero.
    missing = rng.random(len(frame)) < np.where(frame['neighborhood'].eq('East'), .08, .025)
    frame.loc[missing, ['attendance_visits', 'survey_response_count', 'mean_satisfaction']] = np.nan
    return frame

def validate_workshop_data(frame):
    # Check the session grain, categories, finite quantities, and operational constraints.
    assert frame.shape[1] == 14 and frame['session_id'].is_unique
    assert frame['month'].between(1, 24).all() and frame['month'].nunique() == 24
    assert frame['neighborhood'].nunique() == 6 and frame['topic'].nunique() == 4
    assert (frame['capacity_seats'] > 0).all() and (frame['operating_cost_usd'] > 0).all()
    assert frame['registrations'].between(0, frame['capacity_seats']).all()
    observed = frame.dropna(subset=['attendance_visits'])
    assert observed['attendance_visits'].between(0, observed['registrations']).all()
    assert observed['survey_response_count'].between(0, observed['attendance_visits']).all()
    assert (frame['waitlist_count'] >= 0).all()
    assert (frame.loc[frame['waitlist_count'].gt(0), 'registrations'] == frame.loc[frame['waitlist_count'].gt(0), 'capacity_seats']).all()
    assert frame['mean_satisfaction'].dropna().between(1, 5).all()
    assert frame.loc[frame['survey_response_count'].eq(0), 'mean_satisfaction'].isna().all()
    assert frame['attendance_visits'].isna().any()
    assert np.isfinite(frame.select_dtypes('number').to_numpy()[~frame.select_dtypes('number').isna().to_numpy()]).all()

def workshop_summary(frame, by='neighborhood'):
    # Calculate each ratio on the rows where its numerator and denominator are both known.
    result = frame.groupby(by, observed=True).agg(sessions=('session_id', 'size'),
        capacity_all=('capacity_seats', 'sum'), registrations=('registrations', 'sum'),
        waitlist_count=('waitlist_count', 'sum'), operating_cost_usd=('operating_cost_usd', 'sum'))
    observed = frame.dropna(subset=['attendance_visits']).copy()
    observed['session_utilization_pct'] = 100 * observed['attendance_visits'] / observed['capacity_seats']
    known = observed.groupby(by, observed=True).agg(observed_sessions=('session_id', 'size'),
        attendance_visits=('attendance_visits', 'sum'), capacity_observed=('capacity_seats', 'sum'),
        registrations_observed=('registrations', 'sum'), cost_observed=('operating_cost_usd', 'sum'),
        mean_session_utilization_pct=('session_utilization_pct', 'mean'))
    result = result.join(known)
    result['utilization_pct'] = 100 * result['attendance_visits'] / result['capacity_observed']
    result['attendance_rate_pct'] = 100 * result['attendance_visits'] / result['registrations_observed'].replace(0, np.nan)
    result['cost_per_visit_usd'] = result['cost_observed'] / result['attendance_visits'].replace(0, np.nan)
    result['zero_assumption_pct'] = 100 * result['attendance_visits'] / result['capacity_all']
    result['coverage_pct'] = 100 * result['observed_sessions'] / result['sessions']
    responses = observed.dropna(subset=['survey_response_count']).groupby(by, observed=True).agg(
        survey_response_count=('survey_response_count', 'sum'), survey_attendance=('attendance_visits', 'sum'))
    result = result.join(responses)
    result['response_rate_pct'] = 100 * result['survey_response_count'] / result['survey_attendance'].replace(0, np.nan)
    ratings = frame.dropna(subset=['mean_satisfaction', 'survey_response_count']).copy()
    ratings['rating_sum_approx'] = ratings['mean_satisfaction'] * ratings['survey_response_count']
    rating_totals = ratings.groupby(by, observed=True).agg(rating_sum_approx=('rating_sum_approx', 'sum'),
        rating_responses=('survey_response_count', 'sum'), equal_session_rating=('mean_satisfaction', 'mean'))
    result = result.join(rating_totals)
    result['response_weighted_rating'] = result['rating_sum_approx'] / result['rating_responses'].replace(0, np.nan)
    return result


## Step 3: Load and check the data (Run only)
Look at the first rows and the count of missing values in each column. About 2,300 sessions appear; some attendance values are missing on purpose, so you can practice handling unknown values.

In [3]:
# STEP 3: LOAD AND CHECK. Run only.
data = make_workshop_data(STUDENT_INTEGER)
validate_workshop_data(data)

# A smaller table with the columns the charts use. Missing attendance stays missing.
work = pd.DataFrame({'category': data['neighborhood'],
                     'value': data['attendance_visits'],
                     'x': data['capacity_seats'],
                     'period': data['month'],
                     'denominator': data['capacity_seats']})

print('Rows and columns:', data.shape)
display(data.head())
display(data.isna().sum().rename('Missing values').to_frame())

Rows and columns: (2287, 14)


,session_id,month,neighborhood,topic,delivery_format,schedule,duration_hours,capacity_seats,registrations,attendance_visits,waitlist_count,operating_cost_usd,survey_response_count,mean_satisfaction
0,WS-00001,1,North,Digital skills,In person,Weekday evening,2.0,48,48,41.0,9,241.74,21.0,4.62
1,WS-00002,1,North,Digital skills,In person,Weekday daytime,1.0,35,26,20.0,0,185.06,7.0,4.19
2,WS-00003,1,North,Digital skills,In person,Weekend,2.5,20,20,18.0,6,258.68,10.0,3.41
3,WS-00004,1,North,Digital skills,Online,Weekday evening,1.5,62,55,34.0,0,183.21,9.0,4.42
4,WS-00005,1,North,Digital skills,Online,Weekday evening,2.0,90,85,70.0,0,233.48,21.0,3.37


,Missing values
session_id,0
month,0
neighborhood,0
topic,0
delivery_format,0
schedule,0
duration_hours,0
capacity_seats,0
registrations,0
attendance_visits,75


## Step 4: Tableau users: download your data (Edit here)
1. Change `DOWNLOAD_CSV = False` to `True` and run the cell. Colab downloads `workshop_data_<integer>.csv`.
2. In Tableau, choose **Connect → Text file** and open that CSV.
3. Do the same tasks as Steps 5–7 in Tableau. For utilization, use this calculated field:
   `100 * SUM([attendance_visits]) / SUM(IF NOT ISNULL([attendance_visits]) THEN [capacity_seats] END)`
4. Save your workbook, and keep the CSV with it.

Colab users can skip this step.

In [4]:
# STEP 4: TABLEAU DOWNLOAD. Change False to True, then run.
DOWNLOAD_CSV = True

csv_path = OUT / f'workshop_data_{STUDENT_INTEGER}.csv'
data.to_csv(csv_path, index=False)
if DOWNLOAD_CSV:
    from google.colab import files
    files.download(str(csv_path))
    print('Downloading', csv_path.name)
else:
    print('Tableau users: set DOWNLOAD_CSV = True and run this cell again.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 5: Helpers and reference numbers (Run only)
This cell defines the chart helpers and shows two tables:
- a **neighborhood summary** (sessions, attendance, utilization, waitlist, cost per visit);
- **reference numbers** you'll use to check values shown in your charts.

Rates use the **paired denominator**: seats are counted only where attendance is known.

In [ ]:
# STEP 5: HELPERS AND REFERENCE NUMBERS. Run only.

# Total (or average) of 'value' for each group. Groups with no known values stay blank.
def aggregate(frame, by='category', method='sum'):
    grouped = frame.groupby(by, observed=True, dropna=False)['value']
    return grouped.sum(min_count=1) if method == 'sum' else grouped.mean()

# Rate for each group, using only rows where both the value and the denominator are known.
def paired_rates(frame, by='category'):
    known = frame.dropna(subset=['value', 'denominator'])
    result = frame.groupby(by, observed=True, dropna=False).size().rename('total_rows').to_frame()
    result['observed_rows'] = known.groupby(by, observed=True).size().reindex(result.index, fill_value=0)
    result['numerator'] = known.groupby(by, observed=True)['value'].sum(min_count=1)
    result['denominator'] = known.groupby(by, observed=True)['denominator'].sum(min_count=1)
    result['rate_pct'] = 100 * result['numerator'] / result['denominator']
    return result

# Add a title and source line, save the chart to your folder, and show it.
def save_figure(fig, name, audience=AUDIENCE, question=QUESTION, source=f'{SOURCE} | integer {STUDENT_INTEGER}'):
    fig.suptitle(f'{audience}: {question}', fontsize=11, wrap=True)
    fig.text(.01, .01, source, fontsize=7, wrap=True)
    fig.tight_layout(rect=(0, .06, 1, .93))
    fig.savefig(OUT / name)
    plt.show()
    plt.close(fig)

# Axis limits for bar charts that always include zero.
def bar_limits(values):
    finite = pd.Series(values).dropna()
    lo, hi = min(0., float(finite.min())), max(0., float(finite.max()))
    pad = max(.5, (hi - lo) * .25)
    return (lo - pad if lo < 0 else 0, hi + pad)

# Compare numbers shown in a chart with reference numbers. Counts must match exactly;
# other values may differ by rounding to two decimals.
COUNT_KEYS = ('group:', 'selected:', 'observed_value_sum', 'total_waitlist', 'distribution_rows', 'relationship_rows')
def check_displayed(displayed, references, filename):
    if not displayed:
        print('NOT YET CHECKED: enter at least two numbers shown in your charts.')
        return
    unknown = sorted(set(displayed) - set(references))
    if len(displayed) < 2 or unknown:
        raise ValueError(f'Enter at least two numbers with keys from the reference table. Unknown keys: {unknown}')
    checked = pd.DataFrame({'shown_in_chart': pd.Series(displayed, dtype=float)})
    checked['reference'] = pd.Series(references)
    checked['difference'] = checked['shown_in_chart'] - checked['reference']
    checked['allowed'] = [0. if key.startswith(COUNT_KEYS) and float(references[key]).is_integer() else .0051
                          for key in checked.index]
    display(checked)
    if (checked['difference'].abs() > checked['allowed']).any():
        raise ValueError('A number does not match. Check the filter, measure, missing values, and rounding.')
    checked.to_csv(OUT / filename)
    print('Both numbers match the data.')

group_values = aggregate(work)
coverage = work.groupby('category', observed=True)['value'].agg(observed='count', total='size')
rates = paired_rates(work)
neighborhood_summary = workshop_summary(data)
display(neighborhood_summary[['sessions', 'observed_sessions', 'attendance_visits', 'utilization_pct',
                              'waitlist_count', 'cost_per_visit_usd']].round(2))
neighborhood_summary.to_csv(OUT / 'neighborhood_summary.csv')

reference_values = {'observed_value_sum': float(work['value'].sum())}
reference_values.update({f'group:{k}': float(v) for k, v in group_values.dropna().items()})
reference_values.update({f'rate:{k}': float(v) for k, v in rates['rate_pct'].dropna().items()})
reference_values['overall_utilization_pct'] = float(100 * rates['numerator'].sum() / rates['denominator'].sum())
reference_values['total_waitlist'] = float(data['waitlist_count'].sum())
display(pd.Series(reference_values, name='Reference number').rename_axis('Reference key').to_frame().round(4))

## Step 6: Five starting views (Run, then adapt)
1. **Group comparison:** total attendance and utilization can rank neighborhoods differently. Why?
2. **Distribution:** how utilization varies from session to session. What does one bar represent?
3. **Relationship:** attendance against seats. Bigger rooms hold more people, so this pattern is partly built in.
4. **Time:** utilization for your two `COMPARE_CATEGORIES` across months. Gaps are months with unknown attendance.
5. **Small multiples:** the same distribution for each neighborhood, with the same bins and scale, so the panels can be compared.

Change titles, colors, and order to fit your audience and question.

In [ ]:
# STEP 6: FIVE VIEWS. Run, look, then adapt.

# 1. Group comparison: total attendance (left) and utilization (right), same neighborhood order.
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ordered = group_values.sort_values()
axes[0].barh(ordered.index, ordered.values, color='#17628a')
axes[0].set(xlabel=VALUE_LABEL, title='Total attendance')
axes[1].scatter(rates.loc[ordered.index, 'rate_pct'], ordered.index, color='#a64b00')
axes[1].set(xlabel=RATE_LABEL, title='Utilization')
save_figure(fig, 'explore_1_groups.png')

# 2. Distribution of session utilization.
session_rates = 100 * work['value'] / work['denominator']
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(session_rates.dropna(), bins=12, color='#17628a', edgecolor='white')
ax.set(xlabel=RATE_LABEL, ylabel='Sessions', title='Utilization across sessions')
save_figure(fig, 'explore_2_distribution.png')
reference_values['distribution_rows'] = float(session_rates.notna().sum())

# 3. Relationship between seats and attendance.
known = work.dropna(subset=['x', 'value'])
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.scatter(known['x'], known['value'], s=12, alpha=.3, color='#17628a')
ax.set(xlabel='Seats available', ylabel=VALUE_LABEL, title='Seats and attendance for each session')
save_figure(fig, 'explore_3_relationship.png')
reference_values['relationship_rows'] = float(len(known))

# 4. Utilization over time for the two neighborhoods in COMPARE_CATEGORIES.
if not set(COMPARE_CATEGORIES) <= set(work['category']):
    raise ValueError(f'COMPARE_CATEGORIES must use these names: {sorted(work["category"].unique())}')
fig, ax = plt.subplots(figsize=(10, 4.5))
for name, marker in zip(COMPARE_CATEGORIES, ['o', 's']):
    monthly = paired_rates(work.loc[work['category'].eq(name)], by='period')['rate_pct']
    ax.plot(monthly.index, monthly.values, marker=marker, markersize=3, label=name)
ax.set(xlabel='Month', ylabel=RATE_LABEL, title='Utilization by month')
ax.legend(frameon=False)
save_figure(fig, 'explore_4_time.png')

# 5. Small multiples: one panel per neighborhood, shared bins and axes.
names = sorted(work['category'].unique())
fig, axes = plt.subplots(math.ceil(len(names) / 2), 2, figsize=(11, 2.6 * math.ceil(len(names) / 2)),
                         sharex=True, sharey=True, squeeze=False)
edges = np.histogram_bin_edges(session_rates.dropna(), bins=10)
for ax, name in zip(axes.flat, names):
    panel = session_rates.loc[work['category'].eq(name)].dropna()
    ax.hist(panel, bins=edges, color='#17628a', edgecolor='white')
    ax.set(title=f'{name}: {len(panel)} sessions', xlabel=RATE_LABEL, ylabel='Sessions')
for ax in list(axes.flat)[len(names):]:
    ax.set_visible(False)
save_figure(fig, 'explore_5_small_multiples.png')

## Step 7: Plan two revisions (Edit here)
Use the Module 3 graphical-integrity reading and chart checklist. For two of your views, write down:
- which chart, the change you plan, how it helps your audience, and what it gives up.

At least one change should improve accessibility. You'll make these changes in Module 4.

In [ ]:
# STEP 7: YOUR REVISION PLAN. Add two entries, then run.
PLANNED_REVISIONS = []  # e.g. ['View 1: sort bars and add value labels so the coordinator can compare quickly; loses alphabetical lookup.']
print('Revisions planned:', len(PLANNED_REVISIONS))

### Practice checking two numbers (Edit here)
1. Pick two numbers shown in your charts above.
2. Find their keys in the Step 5 reference table, such as `group:North` (total attendance) or `rate:East` (utilization).
3. Type them into `PRACTICE_VALUES` exactly as shown in the chart, and run the cell.

Counts must match exactly; percentages may differ only by rounding to two decimals. Tableau users: compare your numbers with a crosstab instead. This is practice and isn't graded.

In [ ]:
# PRACTICE CHECK. Enter two numbers from your charts, then run.
PRACTICE_VALUES = {}  # e.g. {'group:North': 4210, 'rate:East': 71.35}
check_displayed(PRACTICE_VALUES, reference_values, 'module3_practice_checks.csv')

### Save your Module 3 work
1. Save the notebook (**File → Save**). Your charts and entries stay in the saved copy.
2. Files in the Colab **Files** pane disappear when the session ends. To keep your charts and tables, set `DOWNLOAD_MODULE3_FILES = True` and run the next cell; Colab downloads one ZIP.
3. Tableau users: save your workbook and keep your original sheets.

Nothing is submitted in Module 3. Come back to this same notebook for Phase B in Module 4.

In [ ]:
# SAVE MODULE 3 FILES. Change False to True to download a ZIP of your charts and tables.
DOWNLOAD_MODULE3_FILES = False

(OUT / 'module3_revision_plan.json').write_text(json.dumps(PLANNED_REVISIONS, indent=2), encoding='utf-8')
if DOWNLOAD_MODULE3_FILES:
    zip_path = Path(f'module3_files_integer_{STUDENT_INTEGER}.zip')
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for item in sorted(OUT.iterdir()):
            archive.write(item, arcname=item.name)
    from google.colab import files
    files.download(str(zip_path))
print('Module 3 done. Save this notebook, then continue with Phase B in Module 4.')

## Phase B: Module 4 chart repair and accessible design
Open your saved copy of this notebook. If Colab has restarted, run Steps 1–6 again with the same integer; your Step 7 plan and practice entries are still saved in the cells.

Choose the two views that best answer your question. Make at least two before/after revisions, including at least one accessibility improvement. Use the Module 4 readings.

### Step 8: Two before/after examples (Run, then adapt)
- **Revision 1 (scale):** the "before" bar chart starts the axis above zero, which exaggerates differences. The "after" chart starts at zero.
- **Revision 2 (accessibility):** the "before" chart uses rainbow colors, tilted labels, and no values. The "after" chart is sorted, uses one color, and labels each bar with its value and coverage.

Adapt these to your two chosen views. Keep both versions of each chart, and explain what changed, why it helps your audience, and what it gives up. Write your caption and explanations in the marked lines.

In [ ]:
# STEP 8: BEFORE/AFTER EXAMPLES. Adapt, then write your caption and explanations below.

# Revision 1: the same final-month values with a misleading axis (left) and a zero baseline (right).
final_month = work['period'].max()
selected_values = aggregate(work.loc[work['period'].eq(final_month)]).sort_values()
reference_values.update({f'selected:{k}': float(v) for k, v in selected_values.dropna().items()})
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax in axes:
    ax.barh(selected_values.index, selected_values.values, color='#17628a')
    ax.set(xlabel=VALUE_LABEL, xlim=bar_limits(selected_values))
axes[0].set_xlim(.65 * selected_values.min(), bar_limits(selected_values)[1])
axes[0].set_title(f'Before: axis starts above zero (month {final_month})')
axes[1].set_title(f'After: axis starts at zero (month {final_month})')
for i, v in enumerate(selected_values):
    axes[1].annotate(f'{v:,.0f}', (v, i), xytext=(4, 0), textcoords='offset points', va='center')
save_figure(fig, 'revision_1_scale_before_after.png')

# Revision 2: accessibility.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(group_values.index, group_values.values, color=plt.cm.rainbow(np.linspace(0, 1, len(group_values))))
axes[0].tick_params(axis='x', labelrotation=40)
axes[0].set(title='Before: rainbow colors, tilted labels', ylabel=VALUE_LABEL)
ordered = group_values.sort_values()
axes[1].barh(ordered.index, ordered.values, color='#555555')
for i, (name, v) in enumerate(ordered.items()):
    row = coverage.loc[name]
    axes[1].annotate(f'{v:,.0f}; {row.observed}/{row.total} known', (v, i), xytext=(4, 0),
                     textcoords='offset points', va='center', fontsize=8)
axes[1].set(xlim=bar_limits(ordered), xlabel=VALUE_LABEL, title='After: sorted, one color, labeled with coverage')
save_figure(fig, 'revision_2_accessibility_before_after.png')

# Write your own caption and one explanation per revision.
CORRECTION_CAPTION = ''
REVISION_EXPLANATIONS = []

### Step 9: Compare two ways to average (Run, then explain)
The table shows two utilization figures for each neighborhood:
- `utilization_pct`: seat-weighted, so large sessions count more;
- `mean_session_utilization_pct`: every session counts equally.

If they're close, your conclusion doesn't depend on the choice; say so. If they differ, explain which one fits your audience's question.

In [ ]:
# STEP 9: TWO AVERAGES. Run only.
display(neighborhood_summary[['mean_session_utilization_pct', 'utilization_pct', 'observed_sessions', 'sessions']].round(2))

### Step 10: Check two numbers in your final charts (Edit here)
Enter two numbers shown in your **final** charts, with their reference keys from Step 5. Final-month bar values use `selected:` keys, such as `selected:North`. If you changed a filter or period, add a matching reference number first. Tableau users: keep a crosstab that shows the same two numbers.

In [ ]:
# STEP 10: CHECK TWO NUMBERS. Enter them, then run.
# Optional: add a reference for a different filter, e.g.
# reference_values['custom:North_online'] = data.loc[data['neighborhood'].eq('North') & data['delivery_format'].eq('Online'), 'attendance_visits'].sum()
DISPLAYED_VALUES = {}  # e.g. {'selected:North': 150, 'rate:East': 71.35}
check_displayed(DISPLAYED_VALUES, reference_values, 'module4_checks.csv')

### Step 11: Save and present (Module 4)
**BUS/DSC 363 using course data:** present these Part 1 charts.
**DSC 563, and 363 students doing the +5 option:** these charts are practice. Continue to Part 2 and present your own-data charts instead.

Your 3–4-minute VoiceThread (180 points, due Sunday of Module 4) shows:
- your audience and question;
- two before/after revisions, at least one for accessibility, and why each helps your audience;
- your two checked numbers;
- one idea from a required reading, with its page or section;
- one limitation of the data or the chart.

Explain changes to a measure or filter separately from visual changes. Put your notebook link in a Text comment on the first slide, and share the notebook with the instructor as **Editor** (address in the course sharing guide). If you used AI help, fill in `AI_USE` below.

In [ ]:
# SAVE YOUR MODULE 4 FILES. Fill in AI_USE if you used AI help; change False to True to download a ZIP.
AI_USE = {'used': False, 'tool': '', 'how_it_helped': ''}
DOWNLOAD_MODULE4_FILES = False

notes = {'integer': STUDENT_INTEGER, 'audience': AUDIENCE, 'question': QUESTION,
         'planned_revisions': PLANNED_REVISIONS, 'revision_explanations': REVISION_EXPLANATIONS,
         'caption': CORRECTION_CAPTION, 'ai_use': AI_USE}
(OUT / 'my_notes.json').write_text(json.dumps(notes, indent=2), encoding='utf-8')
if DOWNLOAD_MODULE4_FILES:
    zip_path = Path(f'module4_files_integer_{STUDENT_INTEGER}.zip')
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for item in sorted(OUT.iterdir()):
            archive.write(item, arcname=item.name)
    from google.colab import files
    files.download(str(zip_path))
print('Save this notebook (File → Save) before you record your VoiceThread.')

# Part 2: Your own dataset
**DSC 563 required · BUS/DSC 363 optional (+5 points)**

Here you load a dataset you chose, from your Google Drive, and make the same kind of before/after revisions with it. Before you start:
1. Save your CSV file in your Google Drive.
2. Share it with the instructor.
3. Keep the number of groups readable (about 12 or fewer). If your data has more, filter it before uploading and say what you kept.

**BUS/DSC 363 students not doing the +5 option: you're done. Don't run the cells below.**

In [ ]:
# PART 2 SWITCH. DSC 563: change False to True. BUS/DSC 363: True only for the +5 option.
DO_PART_2 = False

# When Part 2 is off, stop here with a short message so "Run all" doesn't continue.
class PartTwoOff(Exception):
    def _render_traceback_(self):
        return ['Part 2 is off, so the notebook stops here. To do Part 2, change DO_PART_2 to True and run again.']

if not DO_PART_2:
    raise PartTwoOff()
print('Part 2 is on. Continue with Step G1.')

## Step G1: Sign in to Google Drive (Run only)
This is the same sign-in you used in the Data Science Template.

Several popups will appear.

Approve **all** access when Google asks.

In [ ]:
# G1: Packages for loading data from Google Drive, then sign in.
# https://pypi.org/project/PyDrive2/
!pip install -U -q PyDrive2
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

# Google will ask you to approve access. Click "Continue" or "Allow".
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

## Step G2: Load your dataset (Edit here)
Same steps as the Data Science Template:
1. In Google Drive, find your CSV file.
2. Click **Share**, then **Copy link**. The link looks like

`https://drive.google.com/file/d/1WVluSCNJ-RS1zqQ_0EJScPgurw9CmHj/view?usp=sharing`.

3. Copy the file ID: everything between `/d/` and the next `/`. In the example it's `1WVluSCNJ-RS1zqQ_0EJScPgurw9CmHj`.
4. Paste the ID into `file_id` and type the file's name into `file_name`, both inside quotes.
5. Fill in where the data came from and what one row means, then run the cell.

In [ ]:
# G2: LOAD YOUR DATASET. Edit the four values, then run.
file_id = ''  # e.g. '1WVluSCNJ--RS1zqQ_0EJScPgurw9CmHj'
file_name = ''  # e.g. 'my_data.csv'
SOURCE_CITATION = ''  # Publisher, dataset title, URL, and the date you downloaded it.
ROW_MEANING = ''  # What one row is, e.g. 'one county in one year'.

if not file_id or not file_name:
    raise ValueError('Paste your file_id and file_name (in quotes), then run again.')
downloaded = drive.CreateFile({'id': file_id})
downloaded.GetContentFile(file_name)
df = pd.read_csv(file_name)

print(f'{file_name} rows and columns:', df.shape)
display(df.head())
display(df.isna().sum().rename('Missing values').to_frame())

## Options for your own data

The cells below are a starting point. They make the same kind of charts you made with the course data in Part 1, using your columns.

1.   Use them as is: choose your columns in Step G3, then run Steps G4–G6.
2.   Build your own: use your Part 1 code as a guide and write charts that fit your data better. You can ask Gemini (the AI assistant in Colab) for help. Record any AI help in AI_USE.


Either way, you still need two before/after revisions (one for accessibility), an alternative from Step G5, and two checked numbers from Step G6.




## Step G3: Choose your columns, Option 1 (Edit here)
Look at the column names printed above, then fill in:
- `CATEGORY_COLUMN`: the groups to compare (for example, county or product);
- `VALUE_COLUMN`: the number to show (for example, cases or sales);
- `X_COLUMN`: a second number for a relationship chart, or `''`;
- `PERIOD_COLUMN`: a time column (year, month, or date), or `''`;
- `DENOMINATOR_COLUMN`: a column to divide by for a rate (for example, population), or `''`;
- `AGGREGATION`: `'sum'` for counts and amounts, `'mean'` for averages such as scores or prices.

Also write the audience and question for your own data, and the axis label.

In [ ]:
# G3: YOUR COLUMNS. Edit, then run.
CATEGORY_COLUMN = ''
VALUE_COLUMN = ''
X_COLUMN = ''  # optional
PERIOD_COLUMN = ''  # optional
PERIOD_IS_DATE = False  # True if the time column holds dates such as 2024-03-01
DENOMINATOR_COLUMN = ''  # optional
AGGREGATION = 'sum'  # 'sum' or 'mean'
G_VALUE_LABEL = ''  # Axis label with units, e.g. 'Cases (count)'
G_AUDIENCE = ''
G_QUESTION = ''
G_COMPARE = []  # Two groups to compare over time, or [] for the two largest.

# Check the choices and build the table the charts use.
for name in [CATEGORY_COLUMN, VALUE_COLUMN, X_COLUMN, PERIOD_COLUMN, DENOMINATOR_COLUMN]:
    if name and name not in df.columns:
        raise ValueError(f"'{name}' is not a column. Your columns are: {list(df.columns)}")
if not CATEGORY_COLUMN or not VALUE_COLUMN or AGGREGATION not in ('sum', 'mean'):
    raise ValueError("Fill in CATEGORY_COLUMN and VALUE_COLUMN, and use AGGREGATION = 'sum' or 'mean'.")
g_work = pd.DataFrame({'category': df[CATEGORY_COLUMN].astype(str),
                       'value': pd.to_numeric(df[VALUE_COLUMN], errors='coerce')})
if X_COLUMN:
    g_work['x'] = pd.to_numeric(df[X_COLUMN], errors='coerce')
if PERIOD_COLUMN:
    g_work['period'] = pd.to_datetime(df[PERIOD_COLUMN]) if PERIOD_IS_DATE else pd.to_numeric(df[PERIOD_COLUMN], errors='coerce')
if DENOMINATOR_COLUMN:
    g_work['denominator'] = pd.to_numeric(df[DENOMINATOR_COLUMN], errors='coerce').where(lambda d: d > 0)
G_SOURCE = SOURCE_CITATION or file_name
G_VALUE_LABEL = G_VALUE_LABEL or VALUE_COLUMN
newly_blank = int(g_work['value'].isna().sum() - df[VALUE_COLUMN].isna().sum())
if newly_blank:
    print(f'NOTE: {newly_blank} values in {VALUE_COLUMN} are not numbers (for example text) and are treated as unknown.')
g_coverage = g_work.groupby('category')['value'].agg(known='count', total='size')
print(f'{g_work["category"].nunique()} groups in {CATEGORY_COLUMN}.')
if g_work['category'].nunique() > 15:
    print('That is a lot of groups for one chart. Consider filtering your file to the groups your audience needs.')
display(g_coverage)

## Step G4: Two before/after revisions with your data (Run, then adapt)
- **View A, group comparison:** before = rainbow colors, tilted labels, no values; after = sorted, one color, value and coverage labels.
- **View B, over time** (if you have a time column): before = every group in default colors; after = your two `G_COMPARE` groups highlighted with the rest in gray. Without a time column, View B compares a crowded histogram with a clearer one.

When you have a denominator, View B shows a rate (value ÷ denominator × 100) using only rows where both are known. Adapt titles, colors, and order to your audience, and keep both versions of each chart.

In [ ]:
# G4: TWO BEFORE/AFTER VIEWS WITH YOUR DATA. Run, then adapt.
part2 = dict(audience=G_AUDIENCE or AUDIENCE, question=G_QUESTION or QUESTION, source=G_SOURCE)
g_values = aggregate(g_work, method=AGGREGATION)
has_rate = 'denominator' in g_work and g_work['denominator'].notna().any()

# View A: group comparison.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(g_values.index, g_values.values, color=plt.cm.rainbow(np.linspace(0, 1, len(g_values))))
axes[0].tick_params(axis='x', labelrotation=40)
axes[0].set(title='Before: rainbow colors, tilted labels', ylabel=G_VALUE_LABEL)
ordered = g_values.sort_values()
axes[1].barh(ordered.index, ordered.values, color='#17628a')
for i, (name, v) in enumerate(ordered.items()):
    if pd.notna(v):
        row = g_coverage.loc[name]
        axes[1].annotate(f'{v:,.2f}; {row.known}/{row.total} known', (v, i), xytext=(4, 0),
                         textcoords='offset points', va='center', fontsize=8)
axes[1].set(xlim=bar_limits(ordered), xlabel=G_VALUE_LABEL, title='After: sorted, one color, labeled')
save_figure(fig, 'part2_view_a_before_after.png', **part2)

# View B: over time, or a distribution if there is no time column.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
if 'period' in g_work:
    compare = G_COMPARE or list(g_values.sort_values(ascending=False).index[:2])
    if not set(compare) <= set(g_values.index):
        raise ValueError(f'G_COMPARE must use group names from your data: {list(g_values.index)}')
    y_label = 'Rate (%)' if has_rate else G_VALUE_LABEL
    for name in g_values.index:
        part = g_work.loc[g_work['category'].eq(name)]
        series = paired_rates(part, by='period')['rate_pct'] if has_rate else aggregate(part, by='period', method=AGGREGATION)
        axes[0].plot(series.index, series.values, label=name)
        highlighted = name in compare
        axes[1].plot(series.index, series.values, color='#17628a' if name == compare[0] else '#d95f02' if highlighted else '#c8c8c8',
                     linewidth=2.2 if highlighted else 1, label=name if highlighted else None, zorder=3 if highlighted else 1)
    axes[0].legend(fontsize=7, ncol=2, frameon=False)
    axes[0].set(title='Before: every group, default colors', xlabel=PERIOD_COLUMN, ylabel=y_label)
    axes[1].legend(frameon=False)
    axes[1].set(title='After: two groups highlighted, others in gray', xlabel=PERIOD_COLUMN, ylabel=y_label)
else:
    values = g_work['value'].dropna()
    axes[0].hist(values, bins=60)
    axes[0].set(title='Before: too many bins, no labels')
    axes[1].hist(values, bins='auto', color='#17628a', edgecolor='white')
    axes[1].set(title='After: readable bins, labeled axes', xlabel=G_VALUE_LABEL, ylabel='Rows')
save_figure(fig, 'part2_view_b_before_after.png', **part2)

# Write your own caption and one explanation per revision.
G_CAPTION = ''
G_REVISION_EXPLANATIONS = []

## Step G5: Test an alternative (Edit here)
Graduate work includes testing whether your conclusion holds under a reasonable alternative. This starter compares two summaries for each group and reports whether the ranking changes:
- with a denominator: the overall rate (larger rows count more) vs. the average of row-level rates (every row counts equally);
- without one: the total (sum) vs. the average per row (mean).

You may replace it with a comparison that fits your data better: a different filter, time window, grouping, or denominator. Explain what you compared and whether your conclusion changed.

In [ ]:
# G5: ALTERNATIVE. Run, then adapt.
if has_rate:
    known = g_work.dropna(subset=['value', 'denominator']).copy()
    known['row_rate'] = 100 * known['value'] / known['denominator']
    table = pd.DataFrame({'overall_rate': paired_rates(g_work)['rate_pct'],
                          'average_of_row_rates': known.groupby('category')['row_rate'].mean()})
else:
    table = pd.DataFrame({'sum': aggregate(g_work, method='sum'), 'mean': aggregate(g_work, method='mean')})
first, second = table.columns
table['rank_1'] = table[first].rank(ascending=False, method='min')
table['rank_2'] = table[second].rank(ascending=False, method='min')
display(table.round(2))
moved = table.index[table['rank_1'].ne(table['rank_2'])].tolist()
print('Groups whose rank changes:', ', '.join(moved) if moved else 'none: the ranking holds under both summaries')

## Step G6: Check two numbers in your own charts (Edit here)
The table below lists reference numbers for your data: `group:` (the total or average for each group) and, if you have a denominator, `rate:` (each group's rate). Enter two numbers shown in your final charts with their keys, and run.

In [ ]:
# G6: CHECK TWO NUMBERS FROM YOUR DATA. Enter them, then run.
g_reference_values = {f'group:{k}': float(v) for k, v in g_values.dropna().items()}
if has_rate:
    g_reference_values.update({f'rate:{k}': float(v) for k, v in paired_rates(g_work)['rate_pct'].dropna().items()})
display(pd.Series(g_reference_values, name='Reference number').rename_axis('Reference key').to_frame().round(4))

G_DISPLAYED_VALUES = {}  # e.g. {'group:Oneida': 1520, 'rate:Herkimer': 12.4}
check_displayed(G_DISPLAYED_VALUES, g_reference_values, 'part2_checks.csv')

## Step G7: Present your own-data work
**DSC 563:** your VoiceThread presents these own-data charts (not the Part 1 practice charts). **BUS/DSC 363 (+5 option):** present these instead of your Part 1 charts.

Show the same items listed in Step 11, using your own data, plus:
- a citation for your dataset, and a link to the Drive file you shared with the instructor;
- your Step G5 alternative and whether it changed your conclusion;
- **DSC 563:** at least two credible sources: one required course reading and one outside source, labeled **External**.

Save the notebook (**File → Save**) before recording.

### Help with the code
- <a href="https://matplotlib.org/stable/users/explain/quick_start.html" target="_blank" rel="noopener noreferrer">Matplotlib quick-start guide</a>
- <a href="https://pandas.pydata.org/docs/user_guide/missing_data.html" target="_blank" rel="noopener noreferrer">pandas guide to missing data</a>